# Data Owner 1 (DO1) — Aprendizado Federado com `syft-flwr` e Google Drive

Este notebook representa o **Data Owner 1 (DO1)**, responsável por manter **1/3 (partição 0)** do conjunto de dados *Pima Indians Diabetes* de forma privada no Google Colab.

### Pré-verificação Obrigatória (Aba Lateral do Colab)
Antes de começar (ou ao repetir o experimento do zero):
1. Clique no ícone de **Arquivos** (pasta) na aba lateral esquerda do Colab (`/content`).
2. Confirme que **não existem** pastas de execuções anteriores (`SyftBox_*`, `dataset/` ou `outputs/`).
3. Se estiver repetindo o experimento após um erro ou rodada anterior, rode primeiro `do_client.delete_syftbox()` (Etapa 9) e depois clique em **Ambiente de execução → Desconectar e excluir ambiente de execução** para começar com o disco e o Google Drive limpos.

### Ordem de Execução (Grupo: DO1, DO2, DO3 e DS)
> **Atenção:** Cada integrante do grupo deve usar **uma conta Google diferente**.

1. **ETAPA 1 (DO1, DO2, DO3 e DS):** Instalar `syft-flwr` e fazer login com **4 contas Google distintas**.
2. **ETAPA 2 (DS):** Adicionar DO1, DO2 e DO3 como *peers* (`add_peer`).
3. **ETAPA 3 (DO1, DO2 e DO3):** Carregar *peers* e **aprovar o DS** (`approve_peer_request`) até o estado mudar para `ACCEPTED`.
4. **ETAPA 4 (DO1, DO2 e DO3):** Baixar os dados da URL *raw* do GitHub, gerar a partição `2` (1/3 dos dados), **inspecionar seus dados privados vs. mock**, criar o dataset e rodar `sync()`.
5. **ETAPA 5 (DS):** Aguardar ~15s, rodar `ds_client.sync()` e validar os datasets dos 3 DOs.
6. **ETAPA 6 (DS):** Baixar o projeto Flower (do repositório próprio), fazer `bootstrap` e submeter os jobs para DO1, DO2 e DO3.
7. **ETAPA 7 (DO1, DO2 e DO3):** Sincronizar, aprovar o job recebido e executar `process_approved_jobs()`.
8. **ETAPA 8 (DS):** Executar o servidor agregador Flower (`main.py`).
9. **ETAPA 9 (Todos — Opcional):** Limpar os recursos do SyftBox (`delete_syftbox()`).

---
## ETAPA 1 — Instalação (Versão Travada) e Login do DO1
*(Execute em paralelo com DO2, DO3 e DS)*


In [ ]:
# Instalação recomendada via PyPI com versões travadas (mais rápida e segura para turmas grandes)
!uv pip install -q "syft-flwr==0.5.3" "syft-client==0.1.94"

# ==============================================================================
# [PLANO B - RESERVA 1] Instalar de um fork próprio travado em um commit específico:
# !uv pip install -q "git+https://github.com/erickjun321/syft-flwr.git@<HASH_DO_COMMIT>"
#
# [PLANO B - RESERVA 2] Instalar direto do branch main oficial (use apenas se o PyPI falhar):
# !uv pip install -q "git+https://github.com/OpenMined/syft-flwr.git@main"
# ==============================================================================

In [ ]:
import syft_client as sc
import syft_flwr

print(f"syft_client = {sc.__version__}")
print(f"syft_flwr   = {syft_flwr.__version__}")

do_email = input("Digite o e-mail Google do DO1: ").strip()
do_client = sc.login_do(email=do_email)

---
## PAUSA — Aguarde o DS executar a ETAPA 2 (`add_peer`)

Antes de rodar a célula abaixo, confirme que o **Data Scientist (DS)** já executou `ds_client.add_peer` com o e-mail do **DO1**.

---
## ETAPA 3 — Carregar e Aprovar o DS como Peer (Correção Crítica)
*(Execute em DO1, DO2 e DO3 antes de criar o dataset)*

O estado do DS precisa mudar de `PeerState.PENDING` para `PeerState.ACCEPTED`. Criar o dataset antes dessa aprovação impede que o DS enxergue os arquivos *mock* via Google Drive.

In [ ]:
do_client.load_peers()
print("Peers encontrados:", do_client.peers)

# Aprova automaticamente todos os pedidos pendentes (incluindo o DS)
for peer in do_client.peers:
    if str(peer.state).lower().endswith("pending"):
        do_client.approve_peer_request(peer.email)
        print(f"Peer aprovado: {peer.email}")

do_client.load_peers()
do_client.sync()
print("Estado atualizado dos peers:", do_client.peers)

---
## ETAPA 4 — Baixar Dados do GitHub (URL Raw), Particionar (1/3), Inspecionar e Criar Dataset
*(Execute somente após confirmar `PeerState.ACCEPTED` na Etapa 3)*

- **Fonte:** URL *raw* do repositório próprio no GitHub (`pima-indians-diabetes.csv`), salva em cache local (`/content/pima-indians-diabetes.csv`) com retentativa automática (`--retry 3`).
- **Plano B (Reserva):** Se o GitHub estiver inacessível, basta arrastar o arquivo `pima-indians-diabetes.csv` (disponibilizado na pasta da aula) para a aba lateral do Colab (`/content`) e descomentar o bloco de reserva.
- **Particionamento:** Divisão estratificada em **3 partes iguais** (`StratifiedKFold(n_splits=3, shuffle=True, random_state=42)`). Este notebook usa `PARTITION_ID = 0` (exatamente **1/3** das amostras).

In [ ]:
from pathlib import Path
import subprocess
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, train_test_split

# Configurações de particionamento (1/3 dos dados para cada DO)
DATA_URL = "https://raw.githubusercontent.com/erickjun321/Aprendizado-Federado/refs/heads/main/pima-indians-diabetes.csv"
LOCAL_CSV_PATH = Path("/content/pima-indians-diabetes.csv")
NUM_PARTITIONS = 3
PARTITION_ID = 0  # DO1=0, DO2=1, DO3=2
RANDOM_STATE = 42

COLUMNS = [
    "Pregnancies", "Glucose", "BloodPressure", "SkinThickness",
    "Insulin", "BMI", "DiabetesPedigreeFunction", "Age", "Outcome",
]

# 1. Download único em cache local com retentativa automática
if not LOCAL_CSV_PATH.exists():
    subprocess.run(
        ["curl", "-fSL", "--retry", "3", "--retry-delay", "2", DATA_URL, "-o", str(LOCAL_CSV_PATH)],
        check=True,
    )
    print(f"Dataset baixado e salvo em cache: {LOCAL_CSV_PATH}")
else:
    print(f"Usando dataset já presente em: {LOCAL_CSV_PATH}")

# ==============================================================================
# [PLANO B - RESERVA] Upload manual caso o GitHub esteja bloqueado/indisponível:
# 1) Arraste o arquivo 'pima-indians-diabetes.csv' para a aba lateral (/content)
#    ou descomente as 2 linhas abaixo para abrir o seletor de arquivos do Colab:
# from google.colab import files
# uploaded = files.upload()  # envie o arquivo pima-indians-diabetes.csv
# ==============================================================================

df_raw = pd.read_csv(LOCAL_CSV_PATH, header=None)
first_val = str(df_raw.iloc[0, 0]).strip()
if not first_val.replace(".", "", 1).isdigit():
    df = pd.read_csv(LOCAL_CSV_PATH)
else:
    df = df_raw.copy()
    df.columns = COLUMNS

# 2. Particionamento estratificado em 3 partes iguais (1/3 para cada DO)
skf = StratifiedKFold(n_splits=NUM_PARTITIONS, shuffle=True, random_state=RANDOM_STATE)
splits = list(skf.split(df, df["Outcome"]))
_, part_idx = splits[PARTITION_ID]
df_part = df.iloc[part_idx].reset_index(drop=True)

# 3. Divisão local treino/teste (80% / 20% estratificado)
train_df, test_df = train_test_split(
    df_part, test_size=0.2, random_state=RANDOM_STATE, stratify=df_part["Outcome"]
)

# 4. Geração de dados sintéticos (mock) com a mesma estrutura, sem expor registros reais
rng = np.random.default_rng(RANDOM_STATE + PARTITION_ID)

def generate_mock_df(reference_df: pd.DataFrame, n_rows: int) -> pd.DataFrame:
    mock_data = {}
    for col in reference_df.columns:
        if col == "Outcome":
            mock_data[col] = rng.binomial(1, float(reference_df[col].mean()), size=n_rows)
        elif pd.api.types.is_integer_dtype(reference_df[col]):
            low, high = int(reference_df[col].min()), int(reference_df[col].max())
            mock_data[col] = rng.integers(low, high + 1, size=n_rows)
        else:
            low, high = float(reference_df[col].min()), float(reference_df[col].max())
            mock_data[col] = np.round(rng.uniform(low, high, size=n_rows), 3)
    return pd.DataFrame(mock_data)

mock_train_df = generate_mock_df(train_df, len(train_df))
mock_test_df = generate_mock_df(test_df, len(test_df))

# 5. Salva diretórios private/, mock/ e README.md
DATASET_PATH = Path(f"./dataset/pima-indians-diabetes-database-{PARTITION_ID}").expanduser().absolute()
(DATASET_PATH / "private").mkdir(parents=True, exist_ok=True)
(DATASET_PATH / "mock").mkdir(parents=True, exist_ok=True)

train_df.to_csv(DATASET_PATH / "private" / "train.csv", index=False)
test_df.to_csv(DATASET_PATH / "private" / "test.csv", index=False)
mock_train_df.to_csv(DATASET_PATH / "mock" / "train.csv", index=False)
mock_test_df.to_csv(DATASET_PATH / "mock" / "test.csv", index=False)

readme_text = f"""# Pima Indians Diabetes — Partição {PARTITION_ID + 1} de {NUM_PARTITIONS} (DO1)

- **Origem:** GitHub Raw (`pima-indians-diabetes.csv`)
- **Total de amostras na partição:** {len(df_part)} de {len(df)} ({len(df_part)/len(df):.1%})
- **Treino / Teste:** {len(train_df)} / {len(test_df)}
- **Colunas:** {", ".join(df.columns)}
"""
(DATASET_PATH / "README.md").write_text(readme_text, encoding="utf-8")

print(f"DO1 pronto | Total global: {len(df)} | Partição {PARTITION_ID}: {len(df_part)} amostras ({len(df_part)/len(df):.1%})")
print(f"Treino: {len(train_df)} | Teste: {len(test_df)}")

### Inspeção Local dos Dados do DO1 (Privados vs. Mock)
Esta célula permite que o **DO1** visualize uma amostra (`head()`) dos seus **próprios dados privados** (que permanecem inacessíveis ao DS e a DO2, DO3) e compare com os **dados sintéticos (`mock`)** que serão compartilhados com o DS para exploração de esquema.

In [ ]:
from IPython.display import display

print(f"=== [PRIVADO] Amostra dos dados reais de treino do DO1 (inacessível aos demais) ===")
display(train_df.head())

print("\nDistribuição da classe Outcome nos dados privados (treino):")
print(train_df["Outcome"].value_counts().rename(index={0: "Sem diabetes (0)", 1: "Com diabetes (1)"}))

print(f"\n=== [MOCK] Amostra dos dados sintéticos do DO1 (visível ao DS via Drive) ===")
display(mock_train_df.head())

In [ ]:
# Remove dataset anterior se estiver reexecutando a célula
try:
    do_client.datasets.delete("pima-indians-diabetes-database")
except Exception:
    pass

do_client.create_dataset(
    name="pima-indians-diabetes-database",
    mock_path=DATASET_PATH / "mock",
    private_path=DATASET_PATH / "private",
    summary="Partição 1/3 (DO1) do Pima Indians Diabetes Dataset",
    readme_path=DATASET_PATH / "README.md",
    tags=["diabetes", "health", "medicine"],
    sync=True,
)

# Força a sincronização explícita com o Google Drive
do_client.sync()
do_client.datasets.get_all()

---
## PAUSA — Aguarde o DS executar as ETAPAS 5 e 6

Agora o **DS** deve:
1. Aguardar ~15 segundos, sincronizar (`ds_client.sync()`) e explorar os datasets dos 3 DOs (**Etapa 5**).
2. Submeter o job de treinamento federado para DO1, DO2 e DO3 (**Etapa 6**).

---
## ETAPA 7 — Revisar, Aprovar e Executar o Job de Treinamento Federado
*(Execute em DO1, DO2 e DO3 após o DS submeter os jobs na Etapa 6)*

In [ ]:
# Sincroniza para receber o job submetido pelo DS
do_client.sync()
print("Jobs recebidos:", do_client.jobs)
do_client.jobs[0]

In [ ]:
# Aprova o job e inicia o treinamento local nos dados privados
do_client.jobs[0].approve()
print("Status após aprovação:", do_client.jobs)

do_client.process_approved_jobs()

---
## ETAPA 9 — Limpeza (Opcional)
*(Execute somente após o término completo do treinamento federado na Etapa 8 do DS)*

> **Para repetir a experiência do zero:** rode `do_client.delete_syftbox()` abaixo e, em seguida, clique em **Ambiente de execução → Desconectar e excluir ambiente de execução**.

In [ ]:
# Descomente a linha abaixo apenas quando quiser apagar o SyftBox deste DO (local + Google Drive)
# do_client.delete_syftbox()